# Recommending a game to a user

The steps below will take you through how to build your recommendation system. Before attempting this, please make sure you have followed the 03_01_MySQL_Setup.ipynb steps. 

In [11]:
# Uncomment to install packages
# !pip install -r requirements.txt

## Querying MySQL in Python

In [1]:
PASSWORD = '<your_MYSQL_instance_password>'
import mysql.connector

connection = mysql.connector.connect(
  host="localhost",
  user="root",
  passwd=PASSWORD,
  database="steam_data")

In [3]:
sql = 'SELECT count(*) FROM steam_purchase;'
print(sql)

SELECT count(*) FROM steam_purchase;


In [4]:
cursor = connection.cursor()
cursor.execute(sql)
result = cursor.fetchall()
print(result)
connection.close()

[(129511,)]


## Method for querying

In [7]:
import mysql.connector
def query_mysql(query, password, host='localhost', user='root',database='steam_data'):
    connection = mysql.connector.connect(
    host=host,
    user=user,
    passwd=password,
    database=database
    )
    cursor = connection.cursor()
    cursor.execute(query)
    result = cursor.fetchall()
    connection.close()
    return result

In [8]:
result = query_mysql('SELECT count(*) FROM steam_purchase;',
                    password=PASSWORD)
print(result)

[(129511,)]


## Examing the data

In [9]:
result = query_mysql('SELECT * FROM steam_play LIMIT 4;', password=PASSWORD)
for row in result:
	print(row)

('151603712', 'The Elder Scrolls V Skyrim', 'play', 273.0, 0)
('151603712', 'Fallout 4', 'play', 87.0, 0)
('151603712', 'Spore', 'play', 14.9, 0)
('151603712', 'Fallout New Vegas', 'play', 12.1, 0)


In [10]:
purchase_result = query_mysql('SELECT * FROM steam_purchase LIMIT 4;', password=PASSWORD)
for row in purchase_result:
	print(row)

('151603712', 'The Elder Scrolls V Skyrim', 'purchase', 1.0, 0)
('151603712', 'Fallout 4', 'purchase', 1.0, 0)
('151603712', 'Spore', 'purchase', 1.0, 0)
('151603712', 'Fallout New Vegas', 'purchase', 1.0, 0)


## Path based analytics in tabular data

In [11]:

result = query_mysql('SELECT game_name from steam_purchase WHERE id = "151603712";', password=PASSWORD)
print(result[:10])
print(len(result))


[('The Elder Scrolls V Skyrim',), ('Fallout 4',), ('Spore',), ('Fallout New Vegas',), ('Left 4 Dead 2',), ('HuniePop',), ('Path of Exile',), ('Poly Bridge',), ('Left 4 Dead',), ('Team Fortress 2',)]
40


In [12]:
games = ['"' + game[0] + '"' for game in result]
games_string = ','.join(games)
print(games_string)

"The Elder Scrolls V Skyrim","Fallout 4","Spore","Fallout New Vegas","Left 4 Dead 2","HuniePop","Path of Exile","Poly Bridge","Left 4 Dead","Team Fortress 2","Tomb Raider","The Banner Saga","Dead Island Epidemic","BioShock Infinite","Dragon Age Origins - Ultimate Edition","Fallout 3 - Game of the Year Edition","SEGA Genesis & Mega Drive Classics","Grand Theft Auto IV","Realm of the Mad God","Marvel Heroes 2015","Eldevin","Dota 2","BioShock","Robocraft","Garry's Mod","Jazzpunk","Alan Wake","BioShock 2","Fallen Earth","Fallout New Vegas Courier's Stash","Fallout New Vegas Dead Money","Fallout New Vegas Honest Hearts","Grand Theft Auto Episodes from Liberty City","Hitman Absolution","HuniePop Official Digital Art Collection","HuniePop Original Soundtrack","The Banner Saga - Mod Content","The Elder Scrolls V Skyrim - Dawnguard","The Elder Scrolls V Skyrim - Dragonborn","The Elder Scrolls V Skyrim - Hearthfire"


In [13]:
query_2 = f'SELECT id from steam_purchase WHERE game_name ' \
      	f'IN ({games_string}) AND id != "151603712";'
result_2 = query_mysql(query_2, password=PASSWORD)
users = [user[0] for user in result_2]
print(users[:10])
print(len(users))


['187131847', '59945701', '59945701', '59945701', '59945701', '59945701', '59945701', '59945701', '59945701', '53875128']
16223


In [14]:
users = list(set(users))
print(users[:10])
print(len(users))


['197821092', '243808511', '134765677', '233983451', '55906627', '178863914', '213448361', '156941467', '225982565', '49767455']
7728


In [15]:
users = ['"' + user + '"' for user in users]
users_string = ','.join(users)
query_3 = f'SELECT game_name from steam_purchase WHERE id IN ({users_string}) ' \
      	f'AND game_name NOT IN ({games_string});'
 
result_3 = query_mysql(query_3, password=PASSWORD)
recommended_games = [game[0] for game in result_3]
print(recommended_games[:10])


['Ultra Street Fighter IV', 'FINAL FANTASY XIII', "Sid Meier's Civilization V", 'L.A. Noire', 'Company of Heroes Tales of Valor', '7 Days to Die', 'Divekick', 'FINAL FANTASY VII', 'Orcs Must Die! 2', 'Killing Floor']


In [16]:
from collections import Counter
game_frequency = Counter(recommended_games)
print(game_frequency)

Counter({'Unturned': 1120, 'Counter-Strike Global Offensive': 1098, 'Warframe': 733, 'Counter-Strike Source': 589, 'Heroes & Generals': 539, 'Half-Life 2 Lost Coast': 529, 'Portal 2': 506, 'War Thunder': 500, 'Portal': 487, 'Terraria': 436, 'Half-Life 2': 430, 'Counter-Strike': 429, 'Borderlands 2': 414, "Sid Meier's Civilization V": 407, 'PAYDAY 2': 401, 'Half-Life 2 Deathmatch': 393, 'No More Room in Hell': 368, 'Counter-Strike Condition Zero': 366, 'Counter-Strike Condition Zero Deleted Scenes': 366, 'PlanetSide 2': 346, 'Grand Theft Auto San Andreas': 331, 'Half-Life 2 Episode One': 320, 'Half-Life 2 Episode Two': 317, 'Nosgoth': 317, 'Trove': 315, 'Loadout': 310, 'Counter-Strike Nexon Zombies': 292, 'Metro 2033': 285, 'Dirty Bomb': 279, "Tom Clancy's Ghost Recon Phantoms - NA": 277, 'Warface': 269, 'PAYDAY The Heist': 267, 'Neverwinter': 257, 'The Witcher 2 Assassins of Kings Enhanced Edition': 251, 'Saints Row The Third': 249, 'Arma 2': 247, 'Alien Swarm': 244, 'Grand Theft Auto 